In [ ]:
%load_ext autoreload
%autoreload 2

register that environment as a Jupyter kernel:
- python -m ipykernel install --user --name aldar-rag-env --display-name "Python (ALDAR RAG)"

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

print("Project root:", project_root)

In [ ]:
from src.ingestion import load_pdf_directory, extract_pdf_pages

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.ingestion import load_pdf_directory, extract_pdf_pages

In [ ]:
documents = load_pdf_directory("../data/pdfs")

In [ ]:
# documents[0]

In [ ]:
# sample = documents[0]

# print("Source:", sample["source"])
# print("Page:", sample["page"])
# print("\nTEXT:\n")
# print(sample["text"])

In [ ]:
# for doc in documents[:5]:
#     print("=" * 100)
#     print("SOURCE:", doc["source"])
#     print("PAGE:", doc["page"])
#     print("=" * 100)

#     print(doc["text"])

#     print()

## Chunking

In [ ]:
from src.chunking import chunk_pages

In [ ]:
chunks = chunk_pages(documents)

In [ ]:
len(documents), len(chunks)

In [ ]:
chunks[0]

In [ ]:
chunk = chunks[0]

print("Chunk ID:", chunk["chunk_id"])
print("Source:", chunk["source"])
print("Page:", chunk["page"])
print("Chunk:", chunk["chunk_index"])

print("\nTEXT\n")
print(chunk["text"])

In [ ]:
table_chunks = [
    chunk
    for chunk in chunks
    if chunk["source"] == "2022 Q3 AAPL.pdf"
    and chunk["page"] == 4
]

len(table_chunks)

In [ ]:
for chunk in table_chunks:

    print("=" * 100)

    print("Chunk ID:", chunk["chunk_id"])
    print("Page:", chunk["page"])

    print()

    print(chunk["text"])

    print()

In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

In [ ]:
def count_tokens(text):
    return len(encoding.encode(text))

In [ ]:
for chunk in chunks[:10]:

    print(
        chunk["chunk_id"],
        "->",
        count_tokens(chunk["text"]),
        "tokens"
    )

In [ ]:
import pandas as pd

chunk_df = pd.DataFrame(
    [
        {
            "chunk_id": chunk["chunk_id"],
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_index": chunk["chunk_index"],
            "tokens": count_tokens(chunk["text"]),
            "preview": chunk["text"][:150],
        }
        for chunk in chunks
    ]
)

chunk_df.head(20)

In [ ]:
chunks = chunk_pages(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))

In [ ]:
chunk_df["tokens"].describe()

In [ ]:
small_chunks = chunk_df.sort_values("tokens").head(20)

small_chunks[
    ["chunk_id", "source", "page", "tokens", "preview"]
]

In [ ]:
tiny_chunks = [
    chunk
    for chunk in chunks
    if count_tokens(chunk["text"]) < 50
]

print("Tiny chunks:", len(tiny_chunks))

In [ ]:
for chunk in tiny_chunks[:20]:
    print("=" * 100)
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Tokens:", count_tokens(chunk["text"]))
    print()
    print(chunk["text"])

## Embeddings

In [ ]:
# from Archieve.embeddings import generate_embedding, generate_embeddings

# sample_text = chunks[0]["text"]

# embedding = generate_embedding(sample_text)

# print(type(embedding))
# print(len(embedding))
# print(embedding[:10])

In [ ]:
# texts = [chunk["text"] for chunk in chunks[:5]]

# embeddings = generate_embeddings(texts)

# print("Number of embeddings:", len(embeddings))
# print("Embedding dimension:", len(embeddings[0]))

In [ ]:
# from Archieve.embeddings import embed_chunks

# test_embedded_chunks = embed_chunks(
#     chunks[:100],
#     batch_size=50,
# )

In [ ]:
# len(test_embedded_chunks)

In [ ]:
# len(test_embedded_chunks[0]["embedding"])

In [ ]:
# Inspect the structure:
# test_embedded_chunks[0].keys()

In [ ]:
from src.providers.embeddings.factory import (
    get_embedding_provider,
)

embedding_provider = get_embedding_provider()

In [ ]:
embedding_provider.dimension

In [ ]:
query_embedding = embedding_provider.embed_query(
    "What were Apple's total net sales?"
)

len(query_embedding)

In [ ]:
# Test batch embedding:
sample_texts = [
    chunks[0]["text"],
    chunks[1]["text"],
    chunks[2]["text"],
]

vectors = embedding_provider.embed_texts(sample_texts)

print(len(vectors))
print(len(vectors[0]))

In [ ]:
from src.providers.embeddings.factory import (
    get_embedding_provider,
)

from src.providers.vector_store.factory import (
    get_vector_store,
)

In [ ]:
embedding_provider = get_embedding_provider()

vector_store = get_vector_store(
    embedding_dimension=(
        embedding_provider.dimension
    )
)